### Obtener datos de la página WAHIS

En la página web de WAHIS podemos obtener información específica acerca del gusano barrrenador, incluyendo los casos junto con una descripción detallada de cada uno, en esta descripción se detalla las coordenadas geográficas exactas de cada caso en (latitud, longitud), la especie del animal infectado, su estado de salud, la fecha de identificación y una pequeña descripción del animal, esto es fundamental para entrenar los modelos y conocer a detalle la base de datos principal con la que estaremos trabajando.

In [1]:
# Código de prueba para automatizar la descarga en la plataforma de WAHIS 
import requests
import pandas as pd
import time

#  Definimos los IDs de los eventos que queremos descargar
event_ids = [7721, 7772, 7617,7713] 

# Lista vacía donde guardaremos todos los brotes recolectados
todos_los_brotes = []

# Configuramos un agente para que el servidor crea que somos un navegador real
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

#Iniciamos el ciclo para visitar cada URL automáticamente
for event_id in event_ids:
    url = f"https://wahis.woah.org/api/v1/pi/review/event/{event_id}/all-information?language=es"
    
    try:
        print(f"Descargando datos del evento ID: {event_id}...")
        respuesta = requests.get(url, headers=headers)
        
        # Si el servidor responde con éxito (código 200)
        if respuesta.status_code == 200:
            datos_json = respuesta.json()
            
            # Entramos a la lista de "outbreaks" donde se incluye cada caso
            brotes = datos_json.get('outbreaks', [])
            
            for brote in brotes:
                # Extraemos y limpiamos solo la información vital
                info_brote = {
                    "Event_ID": event_id,
                    "Outbreak_ID": brote.get("outbreakId"),
                    "Latitud": brote.get("latitude"),
                    "Longitud": brote.get("longitude"),
                    "Fecha_Inicio": brote.get("startDate"),
                    "Municipio": brote.get("adminDivision"),
                    "Descripcion": brote.get("description")
                }
                todos_los_brotes.append(info_brote)
                
    except Exception as e:
        print(f"Ocurrió un error con el evento {event_id}: {e}")
    
    # Hacemos una pausa de 1 segundo entre cada petición para no saturar ni bloquear el servidor
    time.sleep(1)

# Transformación y Exportación
# Convertimos nuestra lista de diccionarios en un DataFrame de pandas
df = pd.DataFrame(todos_los_brotes)

# Limpiamos los datos: Eliminamos cualquier fila que no tenga coordenadas
df = df.dropna(subset=['Latitud', 'Longitud'])

# Mostramos una vista previa en la consola
print("\n--- Vista previa de los datos extraídos ---")
print(df.head())

# Exportamos a un archivo CSV (utf-8-sig evita que los acentos se rompan en Excel/Power BI)
df.to_csv("Coordenadas_Gusano_Barrenador.csv", index=False, encoding='utf-8-sig')
print("\n¡Éxito! Archivo 'Coordenadas_Gusano_Barrenador.csv' creado y listo para Power BI.")

Descargando datos del evento ID: 7721...
Descargando datos del evento ID: 7772...
Descargando datos del evento ID: 7617...
Descargando datos del evento ID: 7713...

--- Vista previa de los datos extraídos ---
   Event_ID  Outbreak_ID    Latitud    Longitud  \
0      7721       195107  25.204694 -107.555632   
1      7721       195106  25.383233 -107.567211   
2      7721       195105  24.152970 -107.005455   
3      7721       195104  23.853361 -106.779580   
4      7721       195103  24.214922 -106.912038   

                    Fecha_Inicio    Municipio  \
0  2026-07-30T00:00:00.000+00:00     Mocorito   
1  2026-07-28T00:00:00.000+00:00  Badiraguato   
2  2026-07-29T00:00:00.000+00:00        Elota   
3  2026-07-29T00:00:00.000+00:00        Elota   
4  2026-07-29T00:00:00.000+00:00        Elota   

                                         Descripcion  
0    Bovino de 8 días de edad con miasis en ombligo.  
1  Bovino de 3 años de edad con miasis en base de...  
2     Ovino de 4 días de

El resultado de la extracción de datos de prueba ha sido positivo, podemos continuar con nuestra extracción final, el objetivo es obtener los datos reelevantes (Latitud, longitud, Fecha, Municipio, Descripción, Especie, Casos, Muertes) en cada brote filtrando todos los casos registrados en América y concentrarlos en un dataframe en el que podamos trabajar más adelante.

In [3]:
import json
import pandas as pd
import requests
import time

# Lista de datos en la página para extraer los ID´s de cada evento
try:
    with open('lista_eventos.json', 'r', encoding='utf-8') as archivo:
        datos_filtro = json.load(archivo)
    
    # Navegar un nivel hacia adentro, hacia la llave "list"
    lista_real = datos_filtro.get('list', [])
    
    #  Usar 'eventId' para identificar cada evento
    event_ids_crudos = [evento.get('eventId') for evento in lista_real if evento.get('eventId')]
    
    #Eliminar IDs duplicados (convirtiendo la lista a un 'set' matemático y de vuelta a lista)
    event_ids = list(set(event_ids_crudos))
    
    print(f"Se encontraron {len(event_ids_crudos)} reportes, que corresponden a {len(event_ids)} eventos únicos.")
    print(f"Lista de IDs a descargar: {event_ids}\n")
    
except Exception as e:
    print(f"Error fatal al leer lista_eventos.json: {e}")
    # Si falla, el script se detiene en lugar de correr los datos de prevención
    event_ids = []

# Configuración de la petición
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

todos_los_brotes = []
contador = 1
total = len(event_ids)

# Ciclo principal de extracción
for event_id in event_ids:
    url = f"https://wahis.woah.org/api/v1/pi/review/event/{event_id}/all-information?language=es"
    
    try:
        print(f"[{contador}/{total}] Procesando evento ID: {event_id}...")
        respuesta = requests.get(url, headers=headers)
        
        if respuesta.status_code == 200:
            datos_json = respuesta.json()
            
            # Extracción Cuantitativa 
            quantitative_totals = datos_json.get("quantitativeData", {}).get("totals", [])
            
            lista_especies = []
            total_casos = 0
            total_muertes = 0
            
            for q in quantitative_totals:
                nombre_especie = q.get("speciesName", "No especificada")
                casos = q.get("cases", 0)
                muertes = q.get("deaths", 0)
                
                # Formateamos el texto por si hay varias especies (ej. "Bovinos (10), Caprinos (1)")
                lista_especies.append(f"{nombre_especie} ({casos})")
                total_casos += casos
                total_muertes += muertes
                
            especies_str = ", ".join(lista_especies) if lista_especies else "No especificada"
            
            # Extracción Espacial (Coordenadas) 
            brotes = datos_json.get('outbreaks', [])
            
            for brote in brotes:
                info_brote = {
                    "Event_ID": event_id,
                    "Outbreak_ID": brote.get("outbreakId"),
                    "Latitud": brote.get("latitude"),
                    "Longitud": brote.get("longitude"),
                    "Fecha_Inicio": brote.get("startDate"),
                    "Municipio": brote.get("adminDivision"),
                    "Descripcion": brote.get("description", ""),
                    "Especies_Involucradas": especies_str,
                    "Total_Casos_Evento": total_casos,
                    "Total_Muertes_Evento": total_muertes
                }
                todos_los_brotes.append(info_brote)
                
    except Exception as e:
        print(f"  -> Ocurrió un error con el evento {event_id}: {e}")
    
    contador += 1
    # Pausa de 1 segundo (Vital para que el servidor de WAHIS no nos bloquee la IP)
    time.sleep(1)

# Limpieza y Exportación del Dataset
if todos_los_brotes:
    df = pd.DataFrame(todos_los_brotes)
    
    # Limpieza: Borrar filas donde la latitud o longitud estén vacías
    df = df.dropna(subset=['Latitud', 'Longitud'])
    
    # Limpieza: Convertir la columna de fecha a un formato datetime estándar
    df['Fecha_Inicio'] = pd.to_datetime(df['Fecha_Inicio'], errors='coerce').dt.tz_localize(None)
    
    print("\n--- Vista previa del Dataset Final ---")
    print(df[['Latitud', 'Longitud', 'Especies_Involucradas', 'Total_Casos_Evento']].head())
    
    df.to_csv("Dataset_Gusano_Barrenador_Completo.csv", index=False, encoding='utf-8-sig')
    print("\n¡Extracción finalizada! Archivo 'Dataset_Gusano_Barrenador_Completo.csv' creado exitosamente.")
else:
    print("\nNo se pudieron extraer datos.")

Se encontraron 50 reportes, que corresponden a 50 eventos únicos.
Lista de IDs a descargar: [7430, 7431, 5127, 6289, 6163, 6423, 5655, 7328, 7713, 7329, 3109, 7208, 7721, 7210, 7595, 7596, 7209, 6059, 6319, 7216, 680, 7617, 6468, 6981, 6854, 4552, 6857, 6349, 6350, 3792, 7001, 6105, 1242, 7772, 6495, 7519, 6113, 7394, 6883, 5986, 6371, 5864, 7530, 6894, 6513, 5106, 4597, 7159, 2041, 6269]

[1/50] Procesando evento ID: 7430...
[2/50] Procesando evento ID: 7431...
[3/50] Procesando evento ID: 5127...
[4/50] Procesando evento ID: 6289...
[5/50] Procesando evento ID: 6163...
[6/50] Procesando evento ID: 6423...
[7/50] Procesando evento ID: 5655...
  -> Ocurrió un error con el evento 5655: unsupported operand type(s) for +=: 'int' and 'NoneType'
[8/50] Procesando evento ID: 7328...
[9/50] Procesando evento ID: 7713...
[10/50] Procesando evento ID: 7329...
[11/50] Procesando evento ID: 3109...
[12/50] Procesando evento ID: 7208...
[13/50] Procesando evento ID: 7721...
[14/50] Procesando even

In [5]:
import pandas as pd

# Cargar el dataset
df = pd.read_csv("Dataset_Gusano_Barrenador_Completo.csv")

# Diagnóstico: ¿Cuántos Outbreak_ID están repetidos?
duplicados = df.duplicated(subset=['Outbreak_ID']).sum()
print(f"Se encontraron {duplicados} focos duplicados en la base de datos.")

# Limpieza: Eliminar los duplicados conservando solo el primero que aparezca
if duplicados > 0:
    df_limpio = df.drop_duplicates(subset=['Outbreak_ID'], keep='first')
    print(f"Dataset limpio. Pasamos de {len(df)} a {len(df_limpio)} registros únicos.")
    
    # Sobrescribir el archivo limpio
    df_limpio.to_csv("Dataset_Gusano_Barrenador_Unico.csv", index=False)
else:
    df_limpio = df

Se encontraron 0 focos duplicados en la base de datos.


Hemos obtenido la información crucial de los brotes confirmados en zonas de Centroamérica, México y Estados Unidos, posteriormente se hará extracción de variables climatoológicas específicas exactamente en las zonas que se confiraron brotes, esto con el fin de hacer un análisis en Power Bi y generar un informe que pueda detallar más acerca de la plaga y su coomportamiento.